# T13.5 — Stage III -> Stage IV compatibility correction, manual check

**This is a tiny compatibility-verification notebook, not a Stage IV notebook.** No
`c_hat*`, `h`, `pi`, or likelihood is computed anywhere here. It demonstrates two things
against `final_stage_III_IV_specification.txt`:

1. Stage III's stored `death_time` is sufficient, by itself, to determine whether a
   particle should be treated as alive or dead at an arbitrary query time, including one
   strictly between two checkpoints (Sec. 9, Sec. 32's toy example).
2. The weight update's divergence term is the BASE current `div(U)`, not a forcing-
   member-perturbed `div(U_e)` (Sec. 5), confirmed with a controlled field where the two
   are provably different numbers.

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
from iste.environment import GriddedFieldSource, interpolated_divergence
from iste.geometry import metres_per_degree_longitude
from iste.trajectory import run_backward_trajectory
from iste.step import WeightedParticleState, backward_euler_maruyama_step
from iste.forcing_ensemble import ForcingEnsemble, ForcingEnsembleConfig
from iste.ensemble_run import make_perturbed_source

LAT = np.array([10.0, 20.0, 30.0])
LON = np.array([100.0, 105.0, 110.0, 115.0, 120.0, 125.0, 130.0])
TIME = np.datetime64("2020-01-01T00:00", "ns") + np.arange(8) * np.timedelta64(3600, "s")
T0 = TIME[-1]

def make_ocean(uo_grid, vo_grid):
    uo = np.stack([uo_grid] * len(TIME)); vo = np.stack([vo_grid] * len(TIME))
    return GriddedFieldSource(source_path="<synthetic>", time_name="time", time=TIME, lat_name="lat",
                               lat=LAT, lon_name="lon", lon=LON, data={"uo": uo, "vo": vo}, fill_threshold=1e19)

## 1. A particle that dies strictly between two checkpoints

10-minute checkpoints (0..60 min); constant westward speed engineered so the particle
leaves the domain exactly at the 50-minute checkpoint -- the exact toy structure
(`checkpoint 30 min`, `checkpoint 60 min`, `death 50 min`) `final_stage_III_IV_specification.txt`
Sec. 32 describes.

In [ ]:
dlon_per_deg = metres_per_degree_longitude(20.0)
V = 3.0 * dlon_per_deg / 600.0
ocean = make_ocean(np.full((3, 7), V), np.zeros((3, 7)))
k_field = np.zeros((3, 7))
rng = np.random.default_rng(1)
traj = run_backward_trajectory(20.0, 114.0, T0, 6, 600.0, ocean, LAT, LON, k_field, rng)

minutes_back = [(T0 - t) / np.timedelta64(1, "m") for t in traj.timestamps]
print(f"checkpoints (minutes back): {minutes_back}")
print(f"valid:                      {traj.valid.tolist()}")
print(f"death_time (minutes back):  {(T0 - traj.death_time) / np.timedelta64(1, 'm')}")
print(f"death_lat, death_lon:       {traj.death_lat}, {traj.death_lon}")
print(f"death_reason:               {traj.death_reason}")
assert (T0 - traj.death_time) / np.timedelta64(1, "m") == 50.0

## 2. Alive at 45 min, dead at 50 min, excluded after 50 min

Only a plain comparison against the stored `death_time` -- no interpolation, no `c_hat*`,
no Stage IV logic.

In [ ]:
for minutes in [45, 50, 55]:
    query_time = T0 - np.timedelta64(minutes, "m")
    alive = bool(query_time > traj.death_time)  # forward time decreases as theta increases
    print(f"query at {minutes} min (backward) -> alive: {alive}")

assert bool((T0 - np.timedelta64(45, "m")) > traj.death_time)   # alive
assert not bool((T0 - np.timedelta64(50, "m")) > traj.death_time)  # dead (exact)
assert not bool((T0 - np.timedelta64(55, "m")) > traj.death_time)  # dead (after)
print("\nOK -- Stage III's stored death_time alone is sufficient for this determination")

## 3. Weight update uses `div(U)` (base), not `div(U_e)` (perturbed)

In [ ]:
C = 0.001
lon_mesh = LON[None, :] * np.ones((len(LAT), 1))
uo_base = C * (lon_mesh - 110.0)
ocean_base = make_ocean(uo_base, np.zeros((3, 7)))
ensemble = ForcingEnsemble(ocean_base, ForcingEnsembleConfig(
    n_members=2, sigma_u=0.5, spatial_correlation_km=60.0, correlation_time_seconds=3600.0,
    coarse_resolution_deg=0.5, master_seed=1,
))
perturbed = make_perturbed_source(ocean_base, ensemble, member_id=0)

lat_q, lon_q = np.array([20.0]), np.array([115.0])
div_base, _ = interpolated_divergence(ocean_base, lat_q, lon_q, T0)
div_perturbed, _ = interpolated_divergence(perturbed, lat_q, lon_q, T0)
print(f"div(U) [base]:       {div_base[0]:.6e}")
print(f"div(U_e) [perturbed]: {div_perturbed[0]:.6e}")
print(f"different: {not np.isclose(div_base[0], div_perturbed[0])}")

state = WeightedParticleState.initialize(lat0=[20.0], lon0=[115.0])
new_state = backward_euler_maruyama_step(
    state, T0, 900.0, perturbed, LAT, LON, k_field, np.random.default_rng(0), divergence_source=ocean_base,
)
implied_div = -new_state.log_weight[0] / 900.0
print(f"\nadvection uses the PERTURBED field; weight update's implied div = {implied_div:.6e}")
print(f"matches div(U) [base]: {np.isclose(implied_div, div_base[0])}")
assert np.isclose(implied_div, div_base[0])
print("\nOK -- weight update uses div(U), the base field, per the frozen contract's Sec. 5")

## Summary

- Stage III's per-trajectory `death_time`/`death_lat`/`death_lon`/`death_reason` (T13.5)
  give a future Stage IV query everything Sec. 9/32 need to interpolate particle
  position/log-weight between checkpoints and correctly exclude a particle from its
  exact death time onward, without interpolating an already-aggregated `c_hat*` field.
- The weight update's `div(U)` now always uses the base deterministic current, matching
  the frozen contract exactly, even when the particle's own advection uses a forcing-
  member-perturbed `U_e` -- confirmed with a controlled field where the two are provably
  different numbers.
- No Stage IV `c_hat*`/`h`/`pi`/likelihood logic exists anywhere in this repository.